# 被动源面波：真实有限记录互相关

## Goal

本 Notebook 只做被动源。流程为：**集中参数 → 输入图/required 检查 → 离散波数法 Green 传递函数 → 所有噪声源混合 → 接收器 NCF → 相移频散 → 理论曲线 → 保存**。

处理阶段只接收混合后的接收器波形，不使用源编号，也不做逐源理想互相关。`three_component` 模式会用完整 Green 张量先传播 Fx/Fy/Fz，再从混合 X/Y/Z 记录形成 RR/TT/ZZ NCF。日常操作只修改第 2 步的一个 `PARAMETERS` Cell。


## Setup

### 1. 导入同一套 Python/C++ 计算核心


In [1]:
from pathlib import Path
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

from wavedispx.notebook_support import (
    input_error_markdown,
    parameter_reference_markdown,
    prepare_notebook_case,
    required_failure_markdown,
)
from wavedispx.preflight import checks_markdown, plot_preflight_review
from wavedispx.workflow_plotting import (
    plot_workflow_forward,
    plot_workflow_observations,
    plot_workflow_dispersion,
)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('请从 PassiveFW_versionstable 项目内运行本 Notebook')
    PROJECT_ROOT = PROJECT_ROOT.parent
print(f'Project: {PROJECT_ROOT}')


Project: /Volumes/lx_exFAT/PassiveFW_versionstable


## Steps

### 2. 输入参数

只修改下面一个 Cell；模型、receiver、噪声源位置、时间模型、PSD、NCF、频散、拾取和性能参数全部集中。

每个噪声源先传播再在接收器混合：

$$u_i(t)=\sum_s G_{is}(t)*q_s(t),\qquad C_{ij}(\tau)=\langle u_i(t)u_j(t+\tau)\rangle_t.$$

`stationary_gaussian` 在目标 PSD 内生成独立高斯频谱；`poisson_ricker` 使用

$$N_s\sim\mathrm{Poisson}(\lambda T_w),\qquad \tau_n\sim U(0,T_w),\qquad q_s(t)=\sum_n a_nw(t-\tau_n).$$

接收端 NCF 再用 $E(f,c)=|\sum_r\hat C(f,r)e^{i2\pi fr/c}|$ 做相移成像。快速默认 24 源、60 s 只用于程序检查；收敛建议 192–384 源、600–1800 s。

- `input_toml=None`：使用本 Cell；设为路径后整个算例由 TOML 接管。
- `RECEIVER_LAYOUT` 和 `SOURCE_LAYOUT` 可切换 line/grid/file 与 annulus/line/file。
- file 模式的 Receiver、Noise source、PSD 文件例子已放在同一 Cell。


In [2]:
# ==================== 只修改这个参数总控 Cell ====================
INPUT_TOML = None  # 例：'examples/passive_two_layer.toml'；非 None 时忽略下面参数
RECEIVER_LAYOUT = 'line'  # 'line' / 'grid' / 'file'
SOURCE_LAYOUT = 'annulus'  # 'annulus' / 'line' / 'file'
PSD_FILE = None  # PSD_SHAPE='file' 时设为 frequency_hz,power CSV
PSD_SHAPE = 'ricker'  # 'ricker' / 'flat_band' / 'file'
TEMPORAL_MODEL = 'stationary_gaussian'  # 或 'poisson_ricker'
AMBIENT_COMPONENT = 'vertical'  # 'vertical' / 'three_component'

RECEIVER_PRESETS = {
    'line': {
        'layout': 'line', 'start_x_m': -100.0, 'stop_x_m': 100.0,
        'spacing_m': 5.0, 'y_m': 0.0,
    },
    'grid': {
        'layout': 'grid', 'start_x_m': -25.0, 'stop_x_m': 25.0,
        'spacing_x_m': 5.0, 'start_y_m': -25.0, 'stop_y_m': 25.0,
        'spacing_y_m': 5.0, 'distance_bin_m': 5.0,
    },
    'file': {
        'layout': 'file',
        'coordinates_file': 'examples/observation_system/receivers.csv',
        'distance_bin_m': 5.0,
    },
}
SOURCE_PRESETS = {
    'annulus': {
        'layout': 'annulus', 'count': 24, 'seed': 42,
        'inner_radius_m': 250.0, 'outer_radius_m': 350.0,
        'angular_sampling': 'stratified', 'component': AMBIENT_COMPONENT,
    },
    'line': {
        'layout': 'line', 'count': 48, 'seed': 42,
        'x_min_m': -500.0, 'x_max_m': 500.0, 'y_m': 300.0,
        'angular_sampling': 'stratified', 'component': AMBIENT_COMPONENT,
    },
    'file': {
        'layout': 'file', 'count': 48, 'seed': 42,
        'coordinates_file': 'examples/observation_system/noise_sources.csv',
        'angular_sampling': 'stratified', 'component': AMBIENT_COMPONENT,
    },
}

PARAMETERS = {
    'input_toml': INPUT_TOML,
    'template': 'examples/passive_quick.toml',
    'run': {
        'name': 'notebook_passive_quickstart',
        'output_directory': 'output_python/notebook_passive_quickstart',
        'workers': 0, 'frequency_block_size': 8, 'source_block_size': 8,
        'radial_interpolation_step_m': 0.5,
        'store_source_responses': False,
    },
    # 最后一层 thickness_m 必须是 inf；Vp 必须大于 Vs
    'model_layers': [
        {'thickness_m': 10.0, 'vp_mps': 380.0, 'vs_mps': 220.0,
         'density_kgm3': 2000.0, 'qp': float('inf'), 'qs': float('inf')},
        {'thickness_m': float('inf'), 'vp_mps': 760.0, 'vs_mps': 440.0,
         'density_kgm3': 2000.0, 'qp': float('inf'), 'qs': float('inf')},
    ],
    'receivers': RECEIVER_PRESETS[RECEIVER_LAYOUT],
    'ambient_sources': SOURCE_PRESETS[SOURCE_LAYOUT],
    'noise': {
        'temporal_model': TEMPORAL_MODEL, 'seed': 314159,
        'duration_s': 60.0, 'window_s': 10.0, 'dt_s': 0.01,
        'overlap': 0.0, 'psd_shape': PSD_SHAPE, 'psd_file': PSD_FILE,
        'minimum_frequency_hz': 3.0, 'dominant_frequency_hz': 7.0,
        'maximum_frequency_hz': 10.0, 'source_amplitude': 1.0,
        'independent_sources': True, 'event_rate_hz': 0.4,
    },
    'processing': {
        'correlation_normalization': 'spectral_whitening',
        'dispersion_method': 'phase_shift', 'compare_methods': False,
        'frequency_min_hz': 3.5, 'frequency_max_hz': 9.5,
        'velocity_min_mps': 150.0, 'velocity_max_mps': 500.0,
        'velocity_step_mps': 2.0, 'distance_taper_fraction': 0.5,
        'zero_padding': 8.0, 'radial_kernel_lookup_step': 0.002,
        'picker_smoothness': 0.35, 'picker_maximum_jump_mps': 30.0,
        'picker_minimum_energy': 0.05,
        'rayleigh_modes': 3, 'love_modes': 3,
    },
}
# ================================================================


### 3. 构建算例并确认 Vp/Vs/ρ、geometry、PSD 和理论频散


In [3]:
try:
    prepared = prepare_notebook_case(PROJECT_ROOT, 'passive', PARAMETERS)
except (OSError, ValueError) as error:
    display(Markdown(input_error_markdown(error)))
    raise
workflow, config, invalidated = prepared.workflow, prepared.config, prepared.invalidated
print(f'输入来源：{prepared.input_origin}')
display(Markdown(parameter_reference_markdown('passive')))

review = workflow.review_inputs()
display(Markdown(checks_markdown(review)))
display(Markdown(required_failure_markdown(review)))
figure = plot_preflight_review(review)
display(figure)
plt.close(figure)
assert review.ready, 'required 检查失败：按上面的参数位置和“怎么改”修正总控 Cell'


输入来源：参数 Cell + template: /Volumes/lx_exFAT/PassiveFW_versionstable/examples/passive_quick.toml


### 参数详细说明

| 参数路径 | 单位 | 必填/约束 | 作用与调整影响 |
|---|---:|---|---|
| `run.name` | — | 非空 | 算例名；写入图题、manifest 和输出 TOML。 |
| `run.output_directory` | 路径 | 可写目录 | HDF5、CSV、图和配置的输出目录。 |
| `run.workers` | 线程 | 0=自动，≥0 | CPU 并行线程数；正式计算通常用 0。 |
| `run.frequency_block_size` | 频点 | 整数≥1 | C++ 每批频率数；大值更快但更占内存。 |
| `run.source_block_size` | 源 | 整数≥1 | 被动源分块；主动模式影响很小。 |
| `run.radial_interpolation_step_m` | m | >0，且≤最短波长5% | 径向 Green 插值精度；过大会被 required 检查阻止。 |
| `run.store_source_responses` | — | true/false | 是否保存逐噪声源传递函数；通常关闭以节省内存。 |
| `model_layers[].thickness_m` | m | 有限层>0；末层=inf | 层厚；最后一层必须是半空间。 |
| `model_layers[].vp_mps` | m/s | Vp>Vs>0 | P 波速度；影响阻抗与 Rayleigh 椭圆极化。 |
| `model_layers[].vs_mps` | m/s | >0 | S 波速度；控制面波相速度和最短空间波长。 |
| `model_layers[].density_kgm3` | kg/m³ | >0 | 密度；进入层状介质刚度和阻抗。 |
| `model_layers[].qp / qs` | — | >0 或 inf | 品质因子；inf 表示无本征衰减。 |
| `receivers.layout` | — | line/grid/file | 线阵、规则二维面阵或任意二维 CSV。 |
| `receivers.start/stop/spacing` | m | 跨度可被间距整除 | line 使用 x 与 spacing_m；grid 分别使用 x/y。 |
| `receivers.coordinates_file` | CSV | file 时必填 | 表头 x_m,y_m，至少两个不重复有限坐标。 |
| `receivers.distance_bin_m` | m | 被动 grid/file 必填 | 二维台站对转为径向 NCF 道集时的距离箱宽。 |
| `processing.dispersion_method` | — | phase_shift/bessel/hankel | 频散成像核；被动默认 phase_shift，主动默认 hankel。 |
| `processing.compare_methods` | — | true/false | 同时输出三种成像核的同网格对照；默认关闭以节省时间。 |
| `processing.frequency_min/max_hz` | Hz | 0≤fmin<fmax<Nyquist | 频散显示与拾取频带；被动上限不能超过 PSD 上限。 |
| `processing.velocity_min/max_mps` | m/s | 0<vmin<vmax | 相速度扫描范围，必须覆盖预期模态。 |
| `processing.velocity_step_mps` | m/s | >0 | 速度网格步长；更小更细但计算和内存增加。 |
| `processing.distance_taper_fraction` | — | 0–1 | 距离域 Tukey taper；减弱有限孔径截断旁瓣。 |
| `processing.correlation_normalization` | — | spectral_whitening/one_bit/none | NCF 归一化；主动通常 none，被动默认 whitening。 |
| `processing.zero_padding` | 倍 | >0 | 时间 FFT 补零，只细化频率采样，不提高真实分辨率。 |
| `processing.radial_kernel_lookup_step` | — | ≥0 | Bessel/Hankel 核查表步长；0 为逐点精确函数。 |
| `processing.picker_smoothness` | — | ≥0 | 自动拾取连续性惩罚；过大可能压制真实快速变化。 |
| `processing.picker_maximum_jump_mps` | m/s | >0 | 相邻频点允许的最大拾取跳变。 |
| `processing.picker_minimum_energy` | — | 0–1 | 低于该归一化能量的频点不输出拾取。 |
| `processing.rayleigh_modes/love_modes` | 阶 | 整数≥1 | 独立理论频散计算和叠加的模态数。 |
| `ambient_sources.component` | — | vertical/three_component | 竖直快速路径，或物理 Fx/Fy/Fz 经五分量 Green 张量传播并输出 RR/TT/ZZ NCF。 |
| `ambient_sources.layout` | — | annulus/line/file | 源环、道路源线或显式噪声源坐标文件。 |
| `ambient_sources.count` | 源 | 整数≥1 | 参与混合的噪声源数；源环收敛通常建议 192–384。 |
| `ambient_sources.seed` | — | 整数≥0 | 只控制噪声源空间分布，可复现 geometry。 |
| `ambient_sources.angular_sampling` | — | stratified/uniform/random | 源环方位采样；stratified 通常最稳。 |
| `ambient_sources.inner/outer_radius_m` | m | 0≤内半径<外半径 | annulus 源环范围，半径按面积均匀抽样。 |
| `ambient_sources.x_min/x_max/y_m` | m | xmax>xmin | line 源线范围和固定 y。 |
| `ambient_sources.coordinates_file` | CSV | file 时必填 | 显式 noise source 的 x_m,y_m；count 应与文件行数一致。 |
| `noise.temporal_model` | — | stationary_gaussian/poisson_ricker | 平稳高斯连续噪声或具有离散激发时刻的 Poisson 事件。 |
| `noise.seed` | — | 整数≥0 | 控制源时间函数、事件数、时刻、极性和振幅。 |
| `noise.duration_s` | s | >0 | 有限记录总时长；收敛常用 600–1800 s。 |
| `noise.window_s` | s | 0<窗长≤总时长 | 独立合成/互相关窗；还必须覆盖最大台间传播时延。 |
| `noise.dt_s` | s | >0 | 时间采样；PSD 最高频率必须小于 1/(2dt)。 |
| `noise.psd_shape` | — | ricker/flat_band/file | 噪声功率谱形状；file 读取 frequency_hz,power。 |
| `noise.minimum/dominant/maximum_frequency_hz` | Hz | fmin<fdom≤fmax<Nyquist | 噪声能量频带及谱峰。 |
| `noise.psd_file` | CSV | PSD=file 时必填 | 频率递增、功率非负的外部 PSD。 |
| `noise.source_amplitude` | 相对量 | >0 | 单源振幅尺度；影响记录幅值，不改变归一化频散定义。 |
| `noise.independent_sources` | — | 通常 true | 各源独立是接收端 Green 恢复的标准假设；false 仅作相干敏感性实验。 |
| `noise.event_rate_hz` | 1/s/源 | >0 | poisson_ricker 每个源的平均事件率。 |
| `noise.overlap` | — | 当前固定 0 | 独立合成窗不重叠，由流式互相关器直接叠加。 |

| Status | Check | Value | Criterion | Parameter | How to fix |
|---|---|---:|---|---|---|
| ✅ PASS | Temporal Nyquist | 50.00 Hz | > source cutoff 10.00 Hz | noise.dt_s / maximum_frequency_hz | 减小 dt，或降低源/噪声最高频率，使最高频率严格小于 1/(2dt)。 |
| ✅ PASS | Samples per shortest period | 10.0 | >= 8 recommended | noise.dt_s / maximum_frequency_hz | 建议减小 dt，使最高频率对应的最短周期至少有 8 个采样点。 |
| ✅ PASS | Spatial alias estimate | 22.00 Hz | >= processing maximum 9.50 Hz | geometry.receivers spacing / processing.frequency_max_hz | 减小接收点间距或径向距离箱，或降低处理最高频率，直到空间混叠上限覆盖处理频带。 |
| ⚠️ WARN | Aperture at minimum frequency | 1.93 wavelengths | >= 2 wavelengths recommended | geometry.receivers aperture / processing.frequency_min_hz | 建议扩大阵列孔径，或提高处理最低频率，使孔径至少覆盖约 2 个波长。 |
| ✅ PASS | Minimum source-receiver distance | 167.64 m | > 0 m | ambient_sources coordinates / geometry.receivers | 移动与接收器重合的源或炮点；源—接收器最小距离必须大于 0 m。 |
| ✅ PASS | Correlation window | 10.00 s | > required lag 1.31 s | noise.window_s | 把独立窗长增大到大于 1.31 s，并保证不超过总记录时长。 |
| ⚠️ WARN | Ambient source count | 24 | >= 192 recommended for diffuse annular illumination | ambient_sources.count | 源环建议增加到至少 192 个源；正式收敛计算通常使用 192–384 个。 |
| ⚠️ WARN | Maximum source azimuth gap | 22.93° | <= 10° recommended for annular illumination | ambient_sources.count / angular_sampling | 增加噪声源数量并使用 stratified 或 uniform 方位采样，减少照明空缺。 |
| ✅ PASS | Radial Green interpolation step | 0.500 m | <= 1.100 m | run.radial_interpolation_step_m | 把“径向 Green 插值”减小到不超过 1.100 m。 |

✅ **所有 required 检查通过，可以进入正演。**

<Figure size 1700x1000 with 6 Axes>

## Checks

### 4. 可选分步运行

三个开关默认关闭。正式大算例可先只计算传递函数，再决定是否生成长记录 NCF。


In [4]:
RUN_TRANSFER_STAGE = False
if RUN_TRANSFER_STAGE:
    workflow.compute_forward()
    figure = plot_workflow_forward(workflow)
    display(figure)
    plt.close(figure)
    display(workflow.status())


In [5]:
RUN_NCF_STAGE = False
if RUN_NCF_STAGE:
    workflow.simulate_observations()
    figure = plot_workflow_observations(workflow)
    display(figure)
    plt.close(figure)
    display(workflow.status())


In [6]:
RUN_DISPERSION_STAGE = False
if RUN_DISPERSION_STAGE:
    workflow.compute_dispersion()
    figure = plot_workflow_dispersion(workflow)
    display(figure)
    plt.close(figure)
    display(workflow.status())


## Results

### 5. 一键全流程

确认模型、源—接收器分布、PSD 和 required 检查后，只改下面第一行为 `True`。最终汇总图包含 Vp/Vs、ρ、geometry、混合记录/NCF 和频散。


In [7]:
RUN_ALL_NOW = False

if RUN_ALL_NOW:
    artifacts = workflow.run_all(save=True)
    print(f'完成：{artifacts.output_directory}')
    display(Image(filename=str(artifacts.summary_png)))
else:
    print('尚未启动正演。确认输入后把 RUN_ALL_NOW 改为 True。')


尚未启动正演。确认输入后把 RUN_ALL_NOW 改为 True。


## Next Steps

- 快速程序检查：24 源、60 s。
- 可用被动频散：建议至少 192 源、600 s。
- 稳定展示/收敛基准：384 源、1800 s。
- 高阶物理 3C 基准：直接读取 `examples/passive_high_modes_3c.toml`。
- 二维阵列结果是按台间距分箱后的径向平均；方位信息当前不会进入频散图。
